# Native LTX-2.5 A2V lip-sync LoRA training

This Colab runs in order:

1. Configure the runtime.
2. Clone the reviewed fork, install dependencies, read the Colab Secret HF_TOKEN, and download the required LTX-2.5 model files.
3. Audit the 35-clip dataset, create the explicit one-clip holdout, and precompute the 34 training records.
4. Run the three controlled LoRA configurations.

Important: this trains against the LTX-2.5 dev/full transformer. Evaluate it with that same base model; standalone distilled-model compatibility is not assumed. Detailed live diagnostics are limited to the sweep cell. Training loss and optimizer diagnostics are health signals; final lip-sync quality must be judged in the production ComfyUI workflow.

In [ ]:
# @title Configuration and shared helpers

from __future__ import annotations

import datetime as dt
import hashlib
import json
import os
import random
import shlex
import shutil
import subprocess
import sys
import time
from pathlib import Path

# --------- User-editable experiment controls ---------

REPO_URL = "https://github.com/Yuvrajxms09/LTX-2.git"
# Pinned to the verified trainer + dataset revision. Update this deliberately
# when a newer fork revision has been reviewed.
REPO_REF = "728b3da48ce6ea5e2b6509e23e4e65c951d5b8f1"

WORK_ROOT = Path("/content/ltx2.5")
RUNS_ROOT = Path("/content/ltx2.5_runs")
MODEL_ROOT = Path("/content/models/ltx-2.5")
HF_CACHE_ROOT = Path("/content/hf_cache")

RUN_NAME = "a2v_lipsync_lora_sweep_35"
PRECOMPUTE_OVERWRITE = False

# Dataset split. A held-out source clip is used only for validation/inference.
HOLDOUT_SEED = 17
HOLDOUT_COUNT = 1
HOLDOUT_VIDEO_NAME = "citlalli_garcia_rodriguez__yAQkV_agoOE__candidate__6p12s__153f__1280x720.mp4"

# 153 frames = 8 * 19 temporal intervals + 1; VAE-aligned.
VIDEO_BUCKET = "1280x704x153"
WIDTH, HEIGHT, NUM_FRAMES = 1280, 704, 153
FRAME_RATE = 25.0
PREPROCESS_BATCH_SIZE = 1
NUM_DATALOADER_WORKERS = 2
MIN_FREE_DISK_GIB = 80

# --------- Derived paths ---------

REPO_DIR = WORK_ROOT / "LTX-2"
RUN_ROOT = RUNS_ROOT / RUN_NAME
TRAINER_DIR = REPO_DIR / "packages" / "ltx-trainer"
CONFIG_TEMPLATE = TRAINER_DIR / "configs" / "a2v_lipsync_lora.yaml"

SOURCE_DATASET_DIR = REPO_DIR / "datasets" / "training_clips_612"
SOURCE_MANIFEST = SOURCE_DATASET_DIR / "dataset_manifest.jsonl"

RUNTIME_DATASET_DIR = RUN_ROOT / "dataset"
RUNTIME_MEDIA_DIR = RUNTIME_DATASET_DIR / "media"
TRAIN_MANIFEST = RUNTIME_DATASET_DIR / "train_manifest.jsonl"
HOLDOUT_MANIFEST = RUNTIME_DATASET_DIR / "holdout_manifest.jsonl"
SPLIT_SUMMARY = RUNTIME_DATASET_DIR / "split_summary.json"

PRECOMPUTED_DIR = RUN_ROOT / "precomputed"
PREPROCESS_SPEC = PRECOMPUTED_DIR / "preprocess_spec.json"

VALIDATION_DIR = RUN_ROOT / "validation"
VAL_IMAGE = VALIDATION_DIR / "holdout_first_frame.png"
VAL_AUDIO = VALIDATION_DIR / "holdout_audio.wav"


TRANSFORMER_PATH = MODEL_ROOT / "diffusion_models" / "ltx-2.5-22b-dev-transformer-bf16.safetensors"
TEXT_ENCODER_PATH = MODEL_ROOT / "text_encoders" / "gemma4-12b-with-proj-ltx-2.5-bf16.safetensors"
VIDEO_VAE_PATH = MODEL_ROOT / "vae" / "ltx-2.5-video-vae-bf16.safetensors"
AUDIO_VAE_PATH = MODEL_ROOT / "vae" / "ltx-2.5-audio-vae-bf16.safetensors"



def runtime_env(extra: dict[str, str] | None = None) -> dict[str, str]:
    env = os.environ.copy()
    env.update(
        {
            "HF_HOME": str(HF_CACHE_ROOT),
            "CUDA_VISIBLE_DEVICES": "0",
            "TOKENIZERS_PARALLELISM": "false",
            "PYTHONUNBUFFERED": "1",
        }
    )
    if extra:
        env.update(extra)
    return env


def run(
    cmd: list[str | Path],
    *,
    cwd: str | Path | None = None,
    env: dict[str, str] | None = None,
    display_cmd: list[str | Path] | None = None,
) -> subprocess.CompletedProcess[str]:
    cmd = [str(part) for part in cmd]
    shown = [str(part) for part in (display_cmd or cmd)]
    completed = subprocess.run(
        cmd, cwd=str(cwd) if cwd else None, env=runtime_env(env),
        check=False, capture_output=True, text=True,
    )
    if completed.returncode != 0:
        diagnostic = "\n".join(filter(None, [completed.stdout[-8000:], completed.stderr[-8000:]]))
        raise RuntimeError(
            f"Command failed with exit code {completed.returncode}: {shlex.join(shown)}\n{diagnostic}"
        )
    return completed

def check_file(path: Path, *, minimum_bytes: int = 1) -> None:
    if not path.is_file():
        raise FileNotFoundError(f"Required file is missing: {path}")
    if path.stat().st_size < minimum_bytes:
        raise RuntimeError(f"Required file is unexpectedly small: {path} ({path.stat().st_size} bytes)")


def count_pt_files(path: Path) -> int:
    return sum(1 for item in path.rglob("*.pt") if item.is_file())


RUN_ROOT.mkdir(parents=True, exist_ok=True)
HF_CACHE_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
# @title Colab setup, pinned source, authentication, and model download

if not sys.platform.startswith("linux"):
    raise RuntimeError(f"This notebook expects Linux Colab; found {sys.platform!r}.")

if shutil.which("uv") is None:
    print("uv is not installed in this Colab runtime; installing it into the notebook environment.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

if shutil.which("ffmpeg") is None or shutil.which("ffprobe") is None:
    raise RuntimeError("Colab must provide ffmpeg and ffprobe for media validation and first-frame extraction.")

run(["uv", "--version"])
if shutil.which("nvidia-smi"):
    run(["nvidia-smi"])
else:
    raise RuntimeError("nvidia-smi is unavailable; select a GPU runtime before continuing.")

WORK_ROOT.mkdir(parents=True, exist_ok=True)
disk = shutil.disk_usage(WORK_ROOT)
free_gib = disk.free / 2**30
print(f"Free disk at {WORK_ROOT}: {free_gib:.1f} GiB")
if free_gib < MIN_FREE_DISK_GIB:
    raise RuntimeError(
        f"Only {free_gib:.1f} GiB is free. At least {MIN_FREE_DISK_GIB} GiB is required "
        "for the split model pack and preprocessing cache."
    )

REPO_DIR.parent.mkdir(parents=True, exist_ok=True)

if REPO_DIR.exists():
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout; choose a new WORK_ROOT.")
    status = subprocess.check_output(
        ["git", "status", "--porcelain"],
        cwd=REPO_DIR,
        text=True,
    ).strip()
    if status:
        raise RuntimeError(
            f"{REPO_DIR} has uncommitted changes. Resolve them or choose a new WORK_ROOT; "
            "the notebook will not overwrite them."
        )
    run(["git", "fetch", "--no-tags", "origin", REPO_REF], cwd=REPO_DIR)
else:
    run(
        ["git", "clone", "--filter=blob:none", REPO_URL, str(REPO_DIR)],
        cwd=REPO_DIR.parent,
    )
    run(["git", "fetch", "--no-tags", "origin", REPO_REF], cwd=REPO_DIR)

run(["git", "checkout", "--detach", REPO_REF], cwd=REPO_DIR)
actual_ref = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
if actual_ref != REPO_REF:
    raise RuntimeError(f"Checked out {actual_ref}, expected {REPO_REF}.")
run(["git", "status", "--short", "--branch"], cwd=REPO_DIR)

for required_path in (SOURCE_MANIFEST, CONFIG_TEMPLATE, TRAINER_DIR / "scripts" / "train.py"):
    check_file(required_path)
print("Pinned source and dataset are present.")

run(["uv", "sync", "--extra", "natten"], cwd=REPO_DIR)

torch_probe = """
import torch
import ltx_trainer
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available inside the uv environment.")
print("cuda runtime:", torch.version.cuda)
print("device:", torch.cuda.get_device_name(0))
print("device capability:", torch.cuda.get_device_capability(0))
print("bf16 supported:", torch.cuda.is_bf16_supported())
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("The selected GPU does not support bf16, required by this profile.")
props = torch.cuda.get_device_properties(0)
print("GPU VRAM GiB:", round(props.total_memory / 2**30, 1))
print("ltx_trainer import: OK")
"""
run(["uv", "run", "python", "-c", torch_probe], cwd=REPO_DIR)

try:
    from google.colab import userdata
except ImportError as exc:
    raise RuntimeError("This authentication cell must run in Google Colab.") from exc

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Add a Colab secret named HF_TOKEN and enable notebook access for it.")

os.environ["HF_TOKEN"] = hf_token
del hf_token
print("HF_TOKEN loaded from Colab Secrets; direct Hugging Face downloads are enabled.")

MODEL_ROOT.mkdir(parents=True, exist_ok=True)

required_model_files = [
    TRANSFORMER_PATH.relative_to(MODEL_ROOT).as_posix(),
    TEXT_ENCODER_PATH.relative_to(MODEL_ROOT).as_posix(),
    VIDEO_VAE_PATH.relative_to(MODEL_ROOT).as_posix(),
    AUDIO_VAE_PATH.relative_to(MODEL_ROOT).as_posix(),
]
print("Required model files:")
for relative_name in required_model_files:
    print("  ", relative_name)

run(
    [
        "uv",
        "run",
        "hf",
        "download",
        "Lightricks/LTX-2.5",
        *required_model_files,
        "--local-dir",
        str(MODEL_ROOT),
    ],
    cwd=REPO_DIR,
)

for model_path in (TRANSFORMER_PATH, TEXT_ENCODER_PATH, VIDEO_VAE_PATH, AUDIO_VAE_PATH):
    check_file(model_path, minimum_bytes=1024 * 1024)
    print(f"{model_path.name}: {model_path.stat().st_size / 2**30:.2f} GiB")

print("Required LTX-2.5 split pack is ready.")


In [ ]:
# @title Dataset audit and deterministic train/holdout split

def read_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSONL at {path}:{line_number}") from exc
            if not isinstance(row, dict):
                raise ValueError(f"Manifest row {line_number} is not an object.")
            rows.append(row)
    return rows


entries = read_jsonl(SOURCE_MANIFEST)
if not entries:
    raise RuntimeError(f"No entries found in {SOURCE_MANIFEST}.")

required_columns = {"video", "caption"}
for index, entry in enumerate(entries):
    missing = required_columns - entry.keys()
    if missing:
        raise ValueError(f"Manifest row {index} is missing columns: {sorted(missing)}")
    if not str(entry["caption"]).strip():
        raise ValueError(f"Manifest row {index} has an empty caption.")
    source_path = SOURCE_DATASET_DIR / str(entry["video"])
    check_file(source_path, minimum_bytes=1024)
    entry["_source_path"] = str(source_path.resolve())

video_names = [str(entry["video"]) for entry in entries]
if len(video_names) != len(set(video_names)):
    raise ValueError("Manifest contains duplicate video paths.")

if HOLDOUT_VIDEO_NAME is not None:
    matches = [entry for entry in entries if Path(str(entry["video"])).name == HOLDOUT_VIDEO_NAME]
    if len(matches) != 1:
        raise ValueError(f"HOLDOUT_VIDEO_NAME={HOLDOUT_VIDEO_NAME!r} did not identify exactly one entry.")
    holdout_entries = matches
else:
    if not 0 < HOLDOUT_COUNT < len(entries):
        raise ValueError(f"HOLDOUT_COUNT must be between 1 and {len(entries) - 1}.")
    holdout_entries = random.Random(HOLDOUT_SEED).sample(entries, HOLDOUT_COUNT)

holdout_keys = {str(entry["video"]) for entry in holdout_entries}
train_entries = [entry for entry in entries if str(entry["video"]) not in holdout_keys]
if not train_entries:
    raise RuntimeError("The training split is empty.")

# Keep only the public manifest schema in generated manifests.
def runtime_row(entry: dict) -> dict:
    return {
        "video": f"media/{Path(str(entry['video'])).name}",
        "caption": str(entry["caption"]),
    }


RUNTIME_MEDIA_DIR.mkdir(parents=True, exist_ok=True)
all_split_entries = train_entries + holdout_entries
destination_names = [Path(str(entry["video"])).name for entry in all_split_entries]
if len(destination_names) != len(set(destination_names)):
    raise ValueError("Basenames collide when staging the split; rename the source files first.")

for entry in all_split_entries:
    source_path = Path(entry["_source_path"])
    destination = RUNTIME_MEDIA_DIR / source_path.name
    if destination.is_symlink() or destination.exists():
        if destination.is_dir():
            raise RuntimeError(f"Refusing to replace directory: {destination}")
        destination.unlink()
    destination.symlink_to(source_path)

TRAIN_MANIFEST.write_text(
    "\n".join(json.dumps(runtime_row(entry), ensure_ascii=False) for entry in train_entries) + "\n",
    encoding="utf-8",
)
HOLDOUT_MANIFEST.write_text(
    "\n".join(json.dumps(runtime_row(entry), ensure_ascii=False) for entry in holdout_entries) + "\n",
    encoding="utf-8",
)

split_summary = {
    "created_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "source_manifest": str(SOURCE_MANIFEST),
    "selection": "explicit" if HOLDOUT_VIDEO_NAME else "seeded_random",
    "seed": HOLDOUT_SEED,
    "requested_holdout_video": HOLDOUT_VIDEO_NAME,
    "train_count": len(train_entries),
    "holdout_count": len(holdout_entries),
    "train_videos": [str(entry["video"]) for entry in train_entries],
    "holdout_videos": [str(entry["video"]) for entry in holdout_entries],
}
SPLIT_SUMMARY.write_text(json.dumps(split_summary, indent=2) + "\n", encoding="utf-8")

print(f"Dataset audit passed: {len(entries)} clips; train={len(train_entries)}, holdout={len(holdout_entries)}")

def ffprobe_json(path: Path) -> dict:
    completed = subprocess.run(
        [
            "ffprobe",
            "-v",
            "error",
            "-count_frames",
            "-show_entries",
            (
                "format=duration:"
                "stream=index,codec_type,width,height,r_frame_rate,"
                "nb_read_frames,sample_rate,channels"
            ),
            "-of",
            "json",
            str(path),
        ],
        check=False,
        capture_output=True,
        text=True,
    )
    if completed.returncode != 0:
        raise RuntimeError(f"ffprobe failed for {path}: {completed.stderr.strip()}")
    return json.loads(completed.stdout)


def parse_rate(value: str) -> float:
    numerator, denominator = value.split("/", maxsplit=1)
    denominator_value = float(denominator)
    if denominator_value == 0:
        raise ValueError(f"Invalid frame rate: {value}")
    return float(numerator) / denominator_value


for entry in entries:
    path = Path(entry["_source_path"])
    info = ffprobe_json(path)
    streams = info.get("streams", [])
    video_streams = [stream for stream in streams if stream.get("codec_type") == "video"]
    audio_streams = [stream for stream in streams if stream.get("codec_type") == "audio"]

    if len(video_streams) != 1:
        raise ValueError(f"{path.name}: expected one video stream, found {len(video_streams)}")
    if len(audio_streams) != 1:
        raise ValueError(f"{path.name}: expected one audio stream, found {len(audio_streams)}")

    video = video_streams[0]
    audio = audio_streams[0]
    width = int(video.get("width") or 0)
    height = int(video.get("height") or 0)
    fps = parse_rate(str(video.get("r_frame_rate") or "0/1"))
    frames = int(video.get("nb_read_frames") or 0)
    sample_rate = int(audio.get("sample_rate") or 0)
    channels = int(audio.get("channels") or 0)
    duration = float(info.get("format", {}).get("duration") or 0.0)

    problems = []
    if (width, height) != (1280, 720):
        problems.append(f"dimensions={width}x{height}")
    if abs(fps - FRAME_RATE) > 1e-6:
        problems.append(f"fps={fps}")
    if frames != NUM_FRAMES:
        problems.append(f"decoded_frames={frames}")
    if sample_rate != 48000:
        problems.append(f"sample_rate={sample_rate}")
    if channels != 2:
        problems.append(f"channels={channels}")
    if not 6.10 <= duration <= 6.25:
        problems.append(f"container_duration={duration:.3f}s")

    if problems:
        raise ValueError(f"{path.name}: media contract failed: {', '.join(problems)}")

    print(
        f"{path.name}: {width}x{height}, {fps:g} fps, {frames} frames, "
        f"{sample_rate} Hz, {channels} channels, {duration:.3f}s"
    )

print(f"Exact media audit passed for {len(entries)} clips.")


In [ ]:
# @title Holdout first-frame and audio QA

holdout_source = Path(holdout_entries[0]["_source_path"])
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

run(
    [
        "ffmpeg",
        "-hide_banner",
        "-loglevel",
        "error",
        "-y",
        "-i",
        str(holdout_source),
        "-vf",
        r"select=eq(n\,0)",
        "-frames:v",
        "1",
        str(VAL_IMAGE),
    ]
)
run(
    [
        "ffmpeg",
        "-hide_banner",
        "-loglevel",
        "error",
        "-y",
        "-i",
        str(holdout_source),
        "-map",
        "0:a:0",
        "-vn",
        "-ac",
        "2",
        "-ar",
        "48000",
        "-c:a",
        "pcm_s16le",
        str(VAL_AUDIO),
    ]
)
check_file(VAL_IMAGE, minimum_bytes=1024)
check_file(VAL_AUDIO, minimum_bytes=1024)
audio_probe = json.loads(
    subprocess.check_output(
        [
            "ffprobe",
            "-v",
            "error",
            "-select_streams",
            "a:0",
            "-show_entries",
            "stream=channels,sample_rate",
            "-of",
            "json",
            str(VAL_AUDIO),
        ],
        text=True,
    )
)
audio_stream = audio_probe["streams"][0]
if int(audio_stream["channels"]) != 2 or int(audio_stream["sample_rate"]) != 48000:
    raise RuntimeError(f"Validation audio must be stereo 48 kHz for the LTX audio VAE: {audio_stream}")
print("Validation audio audit: stereo, 48000 Hz")

qa_frames = [(0.0, VAL_IMAGE)]
for timestamp in (0.2, 0.4, 1.0):
    qa_path = VALIDATION_DIR / f"qa_{timestamp:.1f}s.png"
    run(
        [
            "ffmpeg",
            "-hide_banner",
            "-loglevel",
            "error",
            "-y",
            "-i",
            str(holdout_source),
            "-ss",
            f"{timestamp:.2f}",
            "-frames:v",
            "1",
            str(qa_path),
        ]
    )
    check_file(qa_path, minimum_bytes=1024)
    qa_frames.append((timestamp, qa_path))

from IPython.display import Audio, Image, display
for timestamp, frame_path in qa_frames:
    print(f"QA frame at {timestamp:.1f}s")
    display(Image(filename=str(frame_path)))
display(Audio(filename=str(VAL_AUDIO)))
print(f"Validation source: {holdout_source.name}")
print(f"Validation image:  {VAL_IMAGE}")
print(f"Validation audio:  {VAL_AUDIO}")


In [ ]:
# @title Precompute and validate training features

PRECOMPUTED_DIR.mkdir(parents=True, exist_ok=True)

def file_signature(path: Path) -> dict:
    stat = path.stat()
    return {"path": str(path), "size": stat.st_size, "mtime_ns": stat.st_mtime_ns}


preprocess_spec = {
    "repo_ref": actual_ref,
    "train_manifest": str(TRAIN_MANIFEST),
    "train_manifest_sha256": hashlib.sha256(TRAIN_MANIFEST.read_bytes()).hexdigest(),
    "resolution_bucket": VIDEO_BUCKET,
    "frame_rate": FRAME_RATE,
    "model_files": {
        "transformer": file_signature(TRANSFORMER_PATH),
        "text_encoder": file_signature(TEXT_ENCODER_PATH),
        "video_vae": file_signature(VIDEO_VAE_PATH),
        "audio_vae": file_signature(AUDIO_VAE_PATH),
    },
}

existing_pt_files = count_pt_files(PRECOMPUTED_DIR)
if PREPROCESS_SPEC.exists():
    saved_spec = json.loads(PREPROCESS_SPEC.read_text(encoding="utf-8"))
    if saved_spec != preprocess_spec and existing_pt_files and not PRECOMPUTE_OVERWRITE:
        raise RuntimeError(
            f"Existing precompute cache at {PRECOMPUTED_DIR} does not match this run. "
            "Use a new RUN_NAME or set PRECOMPUTE_OVERWRITE=True after reviewing the change."
        )
elif existing_pt_files and not PRECOMPUTE_OVERWRITE:
    raise RuntimeError(
        f"Existing precompute files at {PRECOMPUTED_DIR} have no specification. "
        "Use a new RUN_NAME or set PRECOMPUTE_OVERWRITE=True."
    )

PREPROCESS_SPEC.write_text(json.dumps(preprocess_spec, indent=2) + "\n", encoding="utf-8")

preprocess_cmd = [
    "uv",
    "run",
    "python",
    "scripts/process_dataset.py",
    str(TRAIN_MANIFEST),
    "--resolution-buckets",
    VIDEO_BUCKET,
    "--model-path",
    str(TRANSFORMER_PATH),
    "--text-encoder-path",
    str(TEXT_ENCODER_PATH),
    "--video-vae-path",
    str(VIDEO_VAE_PATH),
    "--audio-vae-path",
    str(AUDIO_VAE_PATH),
    "--video-column",
    "video",
    "--caption-column",
    "caption",
    "--output-dir",
    str(PRECOMPUTED_DIR),
    "--device",
    "cuda",
    "--batch-size",
    str(PREPROCESS_BATCH_SIZE),
]
if PRECOMPUTE_OVERWRITE:
    preprocess_cmd.append("--overwrite")

run(preprocess_cmd, cwd=TRAINER_DIR)

expected_count = len(train_entries)
for role in ("latents", "audio_latents", "conditions"):
    role_dir = PRECOMPUTED_DIR / role
    count = count_pt_files(role_dir)
    print(f"{role}: {count} .pt files")
    if count != expected_count:
        raise RuntimeError(f"{role} has {count} items but the training split has {expected_count}.")

print("Precomputed feature audit passed.")

sample_latent = sorted((PRECOMPUTED_DIR / "latents").rglob("*.pt"))[0]
sample_audio = sorted((PRECOMPUTED_DIR / "audio_latents").rglob("*.pt"))[0]
sample_condition = sorted((PRECOMPUTED_DIR / "conditions").rglob("*.pt"))[0]

inspect_code = f"""
from pathlib import Path
import torch

paths = {{
    "latents": Path({str(sample_latent)!r}),
    "audio_latents": Path({str(sample_audio)!r}),
    "conditions": Path({str(sample_condition)!r}),
}}
for name, path in paths.items():
    value = torch.load(path, map_location="cpu", weights_only=True)
    if isinstance(value, dict):
        summary = {{key: (tuple(item.shape) if hasattr(item, "shape") else type(item).__name__) for key, item in value.items()}}
        print(name, path.name, summary)
    else:
        print(name, path.name, type(value).__name__, getattr(value, "shape", None))
"""
run(["uv", "run", "python", "-c", inspect_code], cwd=REPO_DIR)


In [ ]:
# @title Three-configuration LoRA sweep with live diagnostics and HF upload

import csv
import gc
import math
import re
import statistics

from huggingface_hub import HfApi


HF_REPO_ID = "Yuvrajxms09/ltx-2.5-a2v-lipsync-target-scope-35clips"
SWEEP_ROOT = RUNS_ROOT / "a2v_lipsync_target_scope_sweep_35"
CHECKPOINT_INTERVAL = 50
TRACE_EVERY_N_STEPS = 1

A2V_TARGETS = [
    "audio_to_video_attn.to_k",
    "audio_to_video_attn.to_q",
    "audio_to_video_attn.to_v",
    "audio_to_video_attn.to_out.0",
]
VIDEO_SELF_ATTN_TARGETS = [
    "attn1.to_k",
    "attn1.to_q",
    "attn1.to_v",
    "attn1.to_out.0",
]
OFFICIAL_BROAD_TARGETS = ["to_k", "to_q", "to_v", "to_out.0"]

# First isolate adapter scope at one fixed operating point. Rank and LR are
# deliberately held constant so the result answers an architectural question.
SWEEP_CONFIGS = [
    {
        "name": "a2v_only_r16_a16_lr5e-5_s300",
        "target_scope": "audio_to_video_only",
        "target_modules": A2V_TARGETS,
        "rank": 16,
        "alpha": 16,
        "learning_rate": 5.0e-5,
        "steps": 300,
        "purpose": "Causal audio-to-video intervention",
    },
    {
        "name": "a2v_video_self_r16_a16_lr5e-5_s300",
        "target_scope": "audio_to_video_plus_video_self_attention",
        "target_modules": A2V_TARGETS + VIDEO_SELF_ATTN_TARGETS,
        "rank": 16,
        "alpha": 16,
        "learning_rate": 5.0e-5,
        "steps": 300,
        "purpose": "A2V path with mouth-motion capacity",
    },
    {
        "name": "official_broad_r16_a16_lr5e-5_s300",
        "target_scope": "official_broad_attention",
        "target_modules": OFFICIAL_BROAD_TARGETS,
        "rank": 16,
        "alpha": 16,
        "learning_rate": 5.0e-5,
        "steps": 300,
        "purpose": "Official-style multimodal attention baseline",
    },
]

expected_train_count = len(train_entries)
role_counts = {
    role: count_pt_files(PRECOMPUTED_DIR / role)
    for role in ("latents", "audio_latents", "conditions")
}
if any(count != expected_train_count for count in role_counts.values()):
    raise RuntimeError(
        "Precomputed counts do not match the current training split: "
        f"expected={expected_train_count}, actual={role_counts}"
    )

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN is unavailable in the notebook environment.")

api = HfApi(token=hf_token)
api.create_repo(repo_id=HF_REPO_ID, repo_type="model", private=True, exist_ok=True)
SWEEP_ROOT.mkdir(parents=True, exist_ok=True)

ansi_re = re.compile(r"\x1B(?:[@-Z\\-_]|\[[0-?]*[ -/]*[@-~])")
trace_re = re.compile(
    r"Training trace:\s*step=(?P<step>\d+)/(?P<total>\d+),\s*"
    r"loss=(?P<loss>[-+0-9.eE]+),\s*lr=(?P<lr>[-+0-9.eE]+),\s*"
    r"step_time_s=(?P<step_time>[-+0-9.eE]+),\s*"
    r"sigma_range=\((?P<sigma_min>[-+0-9.eE]+),\s*(?P<sigma_max>[-+0-9.eE]+)\)"
    r"(?:,\s*gradient_norm=(?P<gradient_norm>[-+0-9.eE]+))?"
    r"(?:,\s*cuda_allocated_gb=(?P<allocated>[-+0-9.eE]+))?"
    r"(?:,\s*cuda_reserved_gb=(?P<reserved>[-+0-9.eE]+))?"
)


def remove_empty_pretraining_run(path: Path) -> bool:
    if not path.exists():
        return False
    if path.resolve().parent != SWEEP_ROOT.resolve():
        raise RuntimeError(f"Unsafe sweep path refused: {path}")
    if list(path.rglob("*.safetensors")):
        return False
    log_path = path / "training.log"
    if log_path.exists():
        log_text = log_path.read_text(errors="replace")
        if "Training trace:" in log_text or "Starting training" in log_text:
            return False
    shutil.rmtree(path)
    print("Removed empty pre-training run:", path)
    return True


def parse_trace(path: Path) -> list[dict]:
    text = ansi_re.sub("", path.read_text(encoding="utf-8", errors="replace"))
    rows = {}
    for match in trace_re.finditer(text):
        row = {
            "step": int(match.group("step")),
            "total_steps": int(match.group("total")),
            "loss": float(match.group("loss")),
            "learning_rate": float(match.group("lr")),
            "step_time_s": float(match.group("step_time")),
            "sigma_min": float(match.group("sigma_min")),
            "sigma_max": float(match.group("sigma_max")),
            "gradient_norm": float(match.group("gradient_norm")) if match.group("gradient_norm") else None,
            "cuda_allocated_gb": float(match.group("allocated")) if match.group("allocated") else None,
            "cuda_reserved_gb": float(match.group("reserved")) if match.group("reserved") else None,
        }
        row["sigma"] = (row["sigma_min"] + row["sigma_max"]) / 2.0
        rows[row["step"]] = row
    return [rows[step] for step in sorted(rows)]


def summarize_trace(rows: list[dict], expected_steps: int) -> dict:
    observed = [row["step"] for row in rows]
    expected = list(range(1, expected_steps + 1))
    if observed != expected:
        missing = sorted(set(expected) - set(observed))
        raise RuntimeError(
            f"Incomplete training trace: {len(rows)}/{expected_steps}; missing={missing[:10]}"
        )

    losses = [row["loss"] for row in rows]
    gradients = [row["gradient_norm"] for row in rows if row["gradient_norm"] is not None]
    step_times = [row["step_time_s"] for row in rows]
    if not all(math.isfinite(value) for value in losses + gradients + step_times):
        raise RuntimeError("Training metrics contain NaN or infinity.")

    buckets = {"0.00-0.25": [], "0.25-0.50": [], "0.50-0.75": [], "0.75-1.00": []}
    for row in rows:
        sigma = row["sigma"]
        key = (
            "0.00-0.25" if sigma < 0.25 else
            "0.25-0.50" if sigma < 0.50 else
            "0.50-0.75" if sigma < 0.75 else
            "0.75-1.00"
        )
        buckets[key].append(row["loss"])

    bucket_metrics = {
        key: {
            "count": len(values),
            "mean_loss": statistics.mean(values) if values else None,
        }
        for key, values in buckets.items()
    }
    balanced = [
        metric["mean_loss"]
        for metric in bucket_metrics.values()
        if metric["count"] >= 5
    ]

    return {
        "parsed_steps": len(rows),
        "loss": {
            "mean": statistics.mean(losses),
            "median": statistics.median(losses),
            "minimum": min(losses),
            "maximum": max(losses),
            "stdev": statistics.stdev(losses),
            "first_50_mean": statistics.mean(losses[:50]),
            "last_50_mean": statistics.mean(losses[-50:]),
            "sigma_balanced_mean": statistics.mean(balanced) if balanced else None,
        },
        "sigma_buckets": bucket_metrics,
        "gradient_norm": {
            "mean": statistics.mean(gradients) if gradients else None,
            "maximum": max(gradients) if gradients else None,
            "clipped_fraction": (
                sum(value >= 0.999 for value in gradients) / len(gradients)
                if gradients else None
            ),
        },
        "performance": {
            "mean_step_time_s": statistics.mean(step_times),
            "training_minutes": sum(step_times) / 60.0,
            "peak_reserved_gb": max(
                (row["cuda_reserved_gb"] for row in rows if row["cuda_reserved_gb"] is not None),
                default=None,
            ),
        },
    }


def write_metrics_csv(rows: list[dict], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


all_summaries = []
for index, experiment in enumerate(SWEEP_CONFIGS, start=1):
    run_name = experiment["name"]
    run_root = SWEEP_ROOT / run_name
    output_dir = run_root / "outputs"
    checkpoint_dir = output_dir / "checkpoints"
    config_path = run_root / "run_config.yaml"
    metadata_path = run_root / "run_metadata.json"
    log_path = run_root / "training.log"
    metrics_dir = run_root / "metrics"
    final_checkpoint = checkpoint_dir / f"lora_weights_step_{experiment['steps']:05d}.safetensors"

    print(f"\n{'=' * 88}\nExperiment {index}/{len(SWEEP_CONFIGS)}: {run_name}\n{'=' * 88}")

    if not final_checkpoint.exists():
        if run_root.exists():
            if not remove_empty_pretraining_run(run_root):
                raise RuntimeError(
                    f"Partial run at {run_root} contains training work and will not be overwritten."
                )
        run_root.mkdir(parents=True, exist_ok=False)
        metrics_dir.mkdir(parents=True, exist_ok=True)

        config = yaml.safe_load(CONFIG_TEMPLATE.read_text(encoding="utf-8"))
        video_cfg = config["training_strategy"]["video"]
        audio_cfg = config["training_strategy"]["audio"]
        first_frames = [item for item in video_cfg["conditions"] if item["type"] == "first_frame"]
        if not video_cfg["is_generated"] or audio_cfg["is_generated"]:
            raise RuntimeError("Template modality contract is not A2V video generation.")
        if len(first_frames) != 1 or first_frames[0].get("probability") != 1.0:
            raise RuntimeError("Template must always apply first-frame conditioning.")
        if not any(name.startswith("audio_to_video_attn.") for name in config["lora"]["target_modules"]):
            raise RuntimeError("Template has no audio-to-video LoRA targets.")
        if "distilled" in TRANSFORMER_PATH.name.lower():
            raise RuntimeError(
                "This sweep is intentionally trained on the dev/full transformer. "
                "Do not silently substitute a standalone distilled checkpoint."
            )

        config["model"].update({
            "model_path": str(TRANSFORMER_PATH),
            "text_encoder_path": str(TEXT_ENCODER_PATH),
            "video_vae_path": str(VIDEO_VAE_PATH),
            "audio_vae_path": str(AUDIO_VAE_PATH),
            "load_checkpoint": None,
        })
        config["lora"].update({
            "rank": experiment["rank"],
            "alpha": experiment["alpha"],
            "dropout": 0.0,
            "target_modules": list(experiment["target_modules"]),
        })
        config["optimization"].update({
            "learning_rate": experiment["learning_rate"],
            "steps": experiment["steps"],
            "batch_size": 1,
            "gradient_accumulation_steps": 1,
            "max_grad_norm": 1.0,
            "optimizer_type": "adamw",
            "scheduler_type": "linear",
            "enable_gradient_checkpointing": True,
        })
        config["data"].update({
            "preprocessed_data_root": str(PRECOMPUTED_DIR),
            "num_dataloader_workers": NUM_DATALOADER_WORKERS,
        })
        config["validation"].update({
            "samples": [],
            "interval": None,
            "skip_initial_validation": True,
            "generate_audio": False,
            "generate_video": True,
        })
        config["checkpoints"].update({
            "interval": CHECKPOINT_INTERVAL,
            "keep_last_n": -1,
            "precision": "bfloat16",
            "no_resume": True,
            "save_training_state": "minimal",
        })
        config["hub"].update({"push_to_hub": False, "hub_model_id": None})
        config["wandb"]["enabled"] = False
        config["diagnostics"].update({
            "trace_every_n_steps": TRACE_EVERY_N_STEPS,
            "log_batch_shapes": True,
            "log_gradient_stats": True,
            "fail_on_non_finite": True,
        })
        config["seed"] = 42
        config["output_dir"] = str(output_dir)

        serialized = yaml.safe_dump(config, sort_keys=False)
        if "path/to/" in serialized:
            raise RuntimeError(f"Placeholder path remains in {run_name} configuration.")
        config_path.write_text(serialized, encoding="utf-8")

        metadata = {
            "created_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
            "repo_ref": actual_ref,
            "dataset_manifest_sha256": preprocess_spec["train_manifest_sha256"],
            "train_count": expected_train_count,
            "holdout_videos": [str(item["video"]) for item in holdout_entries],
            "experiment": {
                **experiment,
                "target_modules": list(experiment["target_modules"]),
            },
            "checkpoint_interval": CHECKPOINT_INTERVAL,
            "trainer_validation_enabled": False,
            "target_modules": config["lora"]["target_modules"],
        }
        metadata_path.write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")

        validate_code = f"""
from pathlib import Path
import yaml
from ltx_trainer.config import LtxTrainerConfig
path = Path({str(config_path)!r})
validated = LtxTrainerConfig(**yaml.safe_load(path.read_text()))
print('Validated sweep config:', path)
print('rank/alpha/lr/steps:', validated.lora.rank, validated.lora.alpha, validated.optimization.learning_rate, validated.optimization.steps)
print('validation samples:', len(validated.validation.samples))
"""
        run(["uv", "run", "python", "-c", validate_code], cwd=TRAINER_DIR)

        command = ["uv", "run", "python", "scripts/train.py", str(config_path)]
        process_env = runtime_env({"NO_COLOR": "1", "COLUMNS": "500"})
        started = time.monotonic()
        print("Command:", shlex.join(command))
        print("Log:", log_path)
        with log_path.open("w", encoding="utf-8") as log_handle:
            process = subprocess.Popen(
                command,
                cwd=TRAINER_DIR,
                env=process_env,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
            )
            if process.stdout is None:
                process.kill()
                raise RuntimeError("Trainer stdout pipe was not created.")
            for line in process.stdout:
                print(line, end="")
                log_handle.write(line)
                log_handle.flush()
            return_code = process.wait()

        elapsed_minutes = (time.monotonic() - started) / 60.0
        if return_code != 0:
            raise RuntimeError(
                f"Sweep stopped: {run_name} exited {return_code} after {elapsed_minutes:.1f} minutes. "
                f"Inspect {log_path}."
            )
        if not final_checkpoint.exists():
            raise RuntimeError(f"Final checkpoint is missing: {final_checkpoint}")
    else:
        print("Completed run found; training will not be repeated.")

    expected_checkpoints = [
        checkpoint_dir / f"lora_weights_step_{step:05d}.safetensors"
        for step in range(CHECKPOINT_INTERVAL, experiment["steps"] + 1, CHECKPOINT_INTERVAL)
    ]
    missing = [path for path in expected_checkpoints if not path.exists()]
    if missing:
        raise RuntimeError(f"Missing checkpoints for {run_name}: {missing}")
    if not log_path.exists():
        raise RuntimeError(f"Training log is missing: {log_path}")

    rows = parse_trace(log_path)
    summary = summarize_trace(rows, experiment["steps"])
    summary.update({
        "run_name": run_name,
        "target_scope": experiment["target_scope"],
        "target_modules": list(experiment["target_modules"]),
        "rank": experiment["rank"],
        "alpha": experiment["alpha"],
        "learning_rate": experiment["learning_rate"],
        "steps": experiment["steps"],
        "checkpoints": [str(path) for path in expected_checkpoints],
    })
    metrics_dir.mkdir(parents=True, exist_ok=True)
    write_metrics_csv(rows, metrics_dir / "per_step_metrics.csv")
    (metrics_dir / "summary.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
    all_summaries.append(summary)

    print("Run health:", json.dumps(summary["loss"], indent=2))
    try:
        api.upload_folder(
            repo_id=HF_REPO_ID,
            repo_type="model",
            folder_path=str(run_root),
            path_in_repo=f"runs/{run_name}",
            allow_patterns=[
                "run_config.yaml",
                "run_metadata.json",
                "training.log",
                "metrics/*.csv",
                "metrics/*.json",
                "outputs/checkpoints/*.safetensors",
            ],
            commit_message=f"Upload completed sweep run {run_name}",
        )
        print("Uploaded:", run_name)
    except Exception as exc:
        print(f"WARNING: upload failed for {run_name}: {exc}")

    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


summary_path = SWEEP_ROOT / "sweep_results.json"
summary_path.write_text(json.dumps(all_summaries, indent=2) + "\n", encoding="utf-8")
split_copy = SWEEP_ROOT / "dataset_split.json"
shutil.copy2(SPLIT_SUMMARY, split_copy)

for artifact in (summary_path, split_copy, PREPROCESS_SPEC):
    try:
        api.upload_file(
            repo_id=HF_REPO_ID,
            repo_type="model",
            path_or_fileobj=str(artifact),
            path_in_repo=artifact.name,
            commit_message=f"Upload sweep artifact {artifact.name}",
        )
    except Exception as exc:
        print(f"WARNING: upload failed for {artifact.name}: {exc}")

health_order = sorted(
    all_summaries,
    key=lambda item: item["loss"]["sigma_balanced_mean"]
    if item["loss"]["sigma_balanced_mean"] is not None else math.inf,
)
print("\nOptimization-health ordering (not a lip-sync quality ranking):")
for index, summary in enumerate(health_order, start=1):
    print(
        f"{index}. {summary['run_name']}: "
        f"sigma-balanced={summary['loss']['sigma_balanced_mean']}, "
        f"last-50={summary['loss']['last_50_mean']:.6f}"
    )
print("Hugging Face:", f"https://huggingface.co/{HF_REPO_ID}")


## Handoff

The sweep uploads each completed run, checkpoints, per-step metrics, logs, and split metadata to the private Hugging Face repository. Compare checkpoints in ComfyUI using the same prompt, audio, seed, and sampler before selecting a production candidate.